In [ ]:
import requests

query = """
query getAllPokemon {
  pokemon_v2_pokemon(limit: 2000) {
    id
    name
    pokemon_v2_pokemontypes {
      slot
      pokemon_v2_type {
        name
      }
    }
    pokemon_v2_pokemonstats {
      base_stat
      pokemon_v2_stat {
        name
      }
    }
  }
}
"""

response = requests.post(
    'https://beta.pokeapi.co/graphql/v1beta',
    json={'query': query}
)
data = response.json()['data']['pokemon_v2_pokemon']

# 过滤火系并提取所需种族值
fire_pokemon_stats = []
stat_names_map = {
    "attack": "攻击",
    "special-attack": "特攻",
    "hp": "HP",
    "defense": "防御",
    "special-defense": "特防",
    "speed": "速度"
}

for mon in data:
    # 判断是否为火系
    types = [t['pokemon_v2_type']['name'] for t in mon['pokemon_v2_pokemontypes']]
    if 'fire' not in types:
        continue

    # 提取种族值
    stats = {}
    for stat_entry in mon['pokemon_v2_pokemonstats']:
        stat_name = stat_entry['pokemon_v2_stat']['name']
        if stat_name in stat_names_map:
            stats[stat_names_map[stat_name]] = stat_entry['base_stat']

    # 计算总种族值
    total = sum(stats.values())

    fire_pokemon_stats.append({
        'id': mon['id'],
        'name': mon['name'],
        '攻击': stats.get('攻击'),
        '特攻': stats.get('特攻'),
        '总种族值': total
    })

# 输出前5条结果示例
for p in fire_pokemon_stats[:5]:
    print(f"#{p['id']} {p['name']} 攻击:{p['攻击']} 特攻:{p['特攻']} 合计:{p['总种族值']}")

In [ ]:
#导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import mean_squared_error,r2_score

In [ ]:
#设置中文字体
plt.rcParams['font.sans-serif']=['SimHei','Arial Unicode MS','DejaVu Sans']

In [ ]:
df = pd.DataFrame(fire_pokemon_stats)
print("数据前五行：")
print(df.head())
print("\n数据基本信息：")
print(df.info())
print("\n数据统计描述：")
print(df.describe())

In [ ]:
df['最高一攻种族值'] = df[['攻击', '特攻']].max(axis=1)
print(df[['name', '攻击', '特攻', '最高一攻种族值']].head())

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df,x='总种族值',y='最高一攻种族值')
plt.title('火系宝可梦最高一攻种族值与总种族值关系散点图')
plt.xlabel('总种族值')
plt.ylabel('最高一攻种族值')
plt.grid(True,linestyle="--",alpha=0.6)
plt.show()

In [ ]:
X = df[['总种族值']]
y =df[['最高一攻种族值']]
X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.6,
    random_state=42
)

print(f"\n训练集样本数：{len(X_train)}")
print(f"\n测试集样本数：{len(X_test)}")

In [ ]:
model= LinearRegression()
model.fit(X_train,y_train)

In [ ]:
print("模型参数：")
print(f"斜率（系数）：{model.coef_.item():.2f}")
print(f"截距        ：{model.intercept_.item():.2f}")

In [ ]:
y_pred =model.predict(X_test)
mse=mean_squared_error(y_test,y_pred)
mmse = mse/len(X_test)
rmse = np.sqrt(mse)
r2 = r2_score(y_test,y_pred)

In [ ]:
print("\n模型评估结果（在测试集上：")
print(f"均方误差：{mse:.2f}")
print(f"平均均方误差：{mmse:.2f}")
print(f"均方根误差：{rmse:.2f}")
print(f"决定系数：{r2:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))

# 绘制训练集散点
plt.scatter(X_train, y_train, color='blue', alpha=0.6, label='训练集数据')
# 绘制测试集散点
plt.scatter(X_test, y_test, color='green', alpha=0.6, label='测试集数据')

In [ ]:
# 绘制回归线（在特征的全范围内）
X_all = np.linspace(X['总种族值'].min(), X['总种族值'].max(), 100).reshape(-1, 1)
y_all_pred = model.predict(X_all)
plt.plot(X_all, y_all_pred, color='red', linewidth=2, label='回归线')

plt.title('单因子线性回归：总种族值 vs 最高一攻种族值')
plt.xlabel('Total racial value')
plt.ylabel('最高一攻种族值')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
Who = 530    #在这里输入你想要预测的宝可梦的总种族值



new_TotalRacialValue = pd.DataFrame([Who], columns=['总种族值'])   
predicted_HighestAttack = model.predict(new_TotalRacialValue).item()
print(f"\n预测最高一攻种族值为： {predicted_HighestAttack:.2f}")

df['race_diff'] = (df['总种族值'] - Who).abs()   
min_diff = df['race_diff'].min()                    # 最小差值
matched = df[df['race_diff'] == min_diff].copy()    # 可能有多只

if matched.empty:
    print("⚠️ 数据框为空，未找到任何宝可梦。")
else:
    print(f"✅ 总种族值最接近 {Who} 的宝可梦（差值 {min_diff:.0f}），共 {len(matched)} 只：")
    for idx, row in matched.iterrows():
        name = row['name']
        actual_attack = row['最高一攻种族值']   # 确保 df 中已有此列
        diff = actual_attack - predicted_HighestAttack

        print(f"\n🔸 {name} (编号 {row['id']})")
        print(f"   总种族值: {row['总种族值']}")
        print(f"   攻击: {row['攻击']} | 特攻: {row['特攻']} | 最高一攻种族值: {actual_attack}")

        if diff < 0:
            print(f"实际最高一攻种族值比预测低 {abs(diff):.2f} ")
        elif diff > 0:
            print(f"实际最高一攻种族值比预测高 {diff:.2f} ")
        else:
            print(f"实际最高一攻种族值与预测完全一致")